# Testing ADAPT.jl interface in python

In [17]:
import subprocess
import os
from datetime import datetime
from pathlib import Path
import json
from itertools import islice 
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)


In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from src.adapt_utils import (
    run_adapt_jl_parallel,
    show_adapt_logs,
    get_combined_res_df
)

In [20]:
adapt_gpt_dir = Path(
    "/home/mrzaizai2k/code_bao/ADAPT_GPT"
)
adapt_output_dir = "./ADAPT.jl_results/qaoa_python_test"
n_graphs = 5
n_nodes = 10
input_graph_filename = "ADAPT.jl_results/graphs_n10.json"


In [21]:
import networkx as nx
import random

def add_weights_to_nx_graph(G, weighted=True, use_negative=False):
    elist = []
    for u, v in G.edges():
        if weighted:
            w = random.uniform(0.1, 1.0)
            if use_negative and random.random() < 0.5:
                w *= -1
        else:
            w = -1 if (use_negative and random.random() < 0.5) else 1

        elist.append([int(u)+1, int(v)+1, float(round(w, 2))])  
        # +1 to match Julia 1-indexing
    return elist

In [22]:
def generate_graphs(
    n_graphs=10,
    n_nodes=10,
    density=None,          # if None → random
    weighted=True,
    use_negative=False
):
    graphs_dict = {}

    for i in range(n_graphs):
        if density is None:
            p = random.uniform(0.6, 0.9)   # random density
        else:
            p = density

        G = nx.erdos_renyi_graph(n=n_nodes, p=p)

        # avoid empty graph
        while G.number_of_edges() == 0:
            G = nx.erdos_renyi_graph(n=n_nodes, p=p)

        elist = add_weights_to_nx_graph(G, weighted, use_negative)

        graph_name = f"Graph_{i}_n{n_nodes}"

        graphs_dict[graph_name] = {
            "elist": elist,
            "n_nodes": n_nodes
        }

    return graphs_dict

In [23]:
import json

def save_graphs_to_json(graphs_dict, filename):
    with open(filename, "w") as f:
        json.dump(graphs_dict, f, indent=2)

In [24]:
def load_graphs(filename):
    with open(filename, "r") as f:
        return json.load(f)

In [25]:


graphs = generate_graphs(
    n_graphs=n_graphs,
    n_nodes=n_nodes,
    density=None,          # or e.g. 0.7
    weighted=True,
    use_negative=False
)

save_graphs_to_json(graphs, input_graph_filename)

# Load back
cur_input_graphs_dict = load_graphs(input_graph_filename)

# Access like your image
print(cur_input_graphs_dict["Graph_0_n10"])

{'elist': [[1, 2, 0.68], [1, 3, 0.61], [1, 5, 0.89], [1, 6, 0.48], [1, 7, 0.18], [2, 3, 0.43], [2, 7, 0.39], [2, 8, 0.4], [2, 9, 0.91], [2, 10, 0.26], [3, 7, 0.53], [3, 9, 0.2], [4, 6, 0.2], [4, 7, 0.18], [4, 8, 0.17], [4, 10, 0.89], [5, 6, 0.87], [5, 9, 0.77], [6, 8, 0.6], [6, 9, 0.96], [6, 10, 0.34], [7, 9, 0.93], [7, 10, 0.32], [8, 9, 0.32], [8, 10, 0.98]], 'n_nodes': 10}


In [26]:
logs_list, cur_proc = run_adapt_jl_parallel(
    script_dir=adapt_gpt_dir,
    output_dir=adapt_output_dir,
    input_graphs=adapt_gpt_dir / input_graph_filename,
    n_workers=2,
    graphs_number=n_graphs,
    trials_per_graph=1,
    max_params=50,
    gamma_0="gamma0_grid.json",
    pool_name="qaoa_double_pool",
    use_floor_stopper=True,
    temp_folder="temp_data",
)

Splitting input graphs into 2 parts
Starting worker 0 on node: DESKTOP-H2CRQMR
Starting worker 1 on node: DESKTOP-H2CRQMR


In [30]:
show_adapt_logs(logs_list, n_lines=20)

Log: worker_DESKTOP-H2CRQMR_0

Number of callbacks: 6
Exact energy (MQLib): -14.62
Running with optimizer: BFGS
For QAOA reading gamma_0 values from: gamma0_grid.json
For QAOA using gamma_0 = 0.01, trial: 1/1, graph N: 3/3
Success!
final energy:	-14.199804303858848 (through trace)
Took time: 3.15 sec.;
N layers: 10;
g0 = 0.01;
ar = 0.9712588443131908
Good approx ratio achieved with g0 = 0.01, skipping other g0s

Graphs on: unknown; pid: 13378: 100.0%┣██████████████┫ 3/3 [00:44<00:00, 22s/it]

Graphs on: unknown; pid: 13378: 100.0%┣██████████████┫ 3/3 [00:44<00:00, 22s/it]

hostname: unknown; pid: 13378; The script took: 46.29575300216675 seconds.


--------------------------------------------------
Log: worker_DESKTOP-H2CRQMR_1

Number of callbacks: 6
Exact energy (MQLib): -14.88
Running with optimizer: BFGS
For QAOA reading gamma_0 values from: gamma0_grid.json
For QAOA using gamma_0 = 0.01, trial: 1/1, graph N: 2/2
Success!
final energy:	-14.653834684880076 (through trace)
Took time:

In [31]:
show_adapt_logs(logs_list, n_lines=20, pbar_only=True)

Log: worker_DESKTOP-H2CRQMR_0
Graphs on: unknown; pid: 13378: 100.0%┣██████████████┫ 3/3 [00:44<00:00, 22s/it]

Log: worker_DESKTOP-H2CRQMR_1
Graphs on: unknown; pid: 13380: 100.0%┣██████████████┫ 2/2 [01:04<00:00, 64s/it]



In [32]:
combined_res_df = get_combined_res_df(
    adapt_output_dir,
    debug_limit=5,
)

Reading ADAPT.jl results from:
	 /home/mrzaizai2k/code_bao/ADAPT_GPT/ADAPT.jl_results/qaoa_python_test


Opening ADAPT results (qaoa_python_test): 100%|██████████| 4/4 [00:00<00:00, 276.19it/s]


df_list len: 4


Opening graphs (qaoa_python_test): 100%|██████████| 4/4 [00:00<00:00, 1125.31it/s]

df_list len: 4
Graphs count:
g_method
input_file    10
Name: count, dtype: int64
Aggregating results...


In [33]:
combined_res_df.columns

Index(['graph_num', 'run', 'prefix', 'method', 'optimizer', 'gamma0',
       'pooltype', 'graph_name', 'n_nodes', 'energy_list', 'took_time',
       'energy_mqlib', 'op_list', 'approx_ratio', 'edge_weight_norm_coef',
       'β_coeff', 'γ_coeff', 'coeff', 'energy_eigen', 'cut_mqlib', 'cut_adapt',
       'cut_eig', 'state_vect_adapt', 'success_flag', 'g_method',
       'edgelist_json', 'H_frob_norm', 'worker_id', 'edgelist_list',
       'edgelist_list_len', 'num_connected_comp', 'n_layers', 'graph_id'],
      dtype='object')

In [34]:
temp_df = combined_res_df.drop(columns=["prefix", "energy_list", "coeff", "cut_mqlib", "cut_adapt", "cut_eig", "state_vect_adapt", "graph_id"])

In [35]:
temp_df[:1]['approx_ratio']

0    0.970358
Name: approx_ratio, dtype: float64

In [36]:
temp_df[:1]

,graph_num,run,method,optimizer,gamma0,pooltype,graph_name,n_nodes,took_time,energy_mqlib,op_list,approx_ratio,edge_weight_norm_coef,β_coeff,γ_coeff,energy_eigen,success_flag,g_method,edgelist_json,H_frob_norm,worker_id,edgelist_list,edgelist_list_len,num_connected_comp,n_layers
0,1,1,qaoa,BFGS,0.01,qaoa_double_pool,Graph_4_n10,10,7.495096,-12.7,"[159, 126, 15, 107, 174, 27, 105, 156, 11, 6, 158]",0.970358,1.0,"[2.2602598166096253, 0.7856744325549011, 0.7856755234745699, 0.787152014505671, -0.7944053018033658, -0.7852018620754003, -0.7815072110802892, -0.7858173926333795, -0.0040513990208459, 0.7416641765889828, 0.7745730664396397]","[-0.0095932537323816, 0.0008217379755325, 0.0132399775080316, 0.0017352864373984, 0.027181407556836, -0.121839598549184, 1.3371113097775384, -0.814350531857266, 2.2032899412987805, -2.086573582918875, 1.7748614734687926]",-12.7,True,input_file,"[[1,2,0.92],[2,3,0.81],[2,4,0.76],[3,4,0.21],[1,5,0.63],[2,5,0.81],[3,5,0.4],[1,6,0.65],[2,6,0.43],[3,6,0.2],[4,6,0.31],[5,6,0.44],[1,7,0.22],[2,7,0.26],[3,7,0.66],[4,7,0.42],[6,7,0.82],[1,8,0.85],[2,8,0.12],[4,8,0.32],[5,8,0.5],[6,8,0.98],[7,8,0.81],[3,9,0.28],[4,9,0.95],[5,9,0.18],[6,9,0.36],[8,9,0.3],[1,10,0.58],[2,10,0.15],[3,10,0.87],[4,10,0.33],[5,10,0.41],[7,10,0.64],[8,10,0.2],[9,10,0.49]]",297.397264,worker_unknown_pid_11930_ts_26-03-27__14_17_graphs_json,"[[1, 2, 0.92], [2, 3, 0.81], [2, 4, 0.76], [3, 4, 0.21], [1, 5, 0.63], [2, 5, 0.81], [3, 5, 0.4], [1, 6, 0.65], [2, 6, 0.43], [3, 6, 0.2], [4, 6, 0.31], [5, 6, 0.44], [1, 7, 0.22], [2, 7, 0.26], [3, 7, 0.66], [4, 7, 0.42], [6, 7, 0.82], [1, 8, 0.85], [2, 8, 0.12], [4, 8, 0.32], [5, 8, 0.5], [6, 8, 0.98], [7, 8, 0.81], [3, 9, 0.28], [4, 9, 0.95], [5, 9, 0.18], [6, 9, 0.36], [8, 9, 0.3], [1, 10, 0.58], [2, 10, 0.15], [3, 10, 0.87], [4, 10, 0.33], [5, 10, 0.41], [7, 10, 0.64], [8, 10, 0.2], [9, 10, 0.49]]",36,1,11


We can read graph from edgelist_json, use our model to create data and comapre with approx_ratio, compare n_layers